# Multimodal Deepfake Detection: Video Swin Tiny + WavLM-Base+ + Cross-Attention Fusion

This notebook implements the new research direction for the FakeBDTeen dataset.

- Pilot target: 970 clips for fast validation and architecture proof
- Full target: scale to the complete dataset with the same staged training plan
- Modality design: Video Swin Tiny for video, WavLM-Base+ for audio, and cross-attention fusion for 4-class classification

Training stages are built into the notebook plan:
- Stage A: train unimodal branches with frozen backbones
- Stage B: unfreeze top blocks and fine-tune branches
- Stage C: train fusion with partially frozen encoders
- Stage D: end-to-end low learning-rate fine-tuning

The notebook uses speaker-aware splitting to reduce leakage, class-balanced sampling to handle the weaker FR class, mixed precision, gradient accumulation, and augmentation hooks for both modalities.

In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import subprocess
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms

try:
    import timm
except ImportError:
    timm = None

try:
    from transformers import AutoFeatureExtractor, WavLMModel
except ImportError:
    AutoFeatureExtractor = None
    WavLMModel = None

print('Imports ready')

In [ ]:
PROJECT_ROOT = Path(r'F:\Sixth Semester\DEEPfake Papers\FakeBDTeen')
RAW_DATASET_ROOT = PROJECT_ROOT / 'FakeBDTeen'
WORK_DIR = PROJECT_ROOT / 'Outputs' / 'video_swin_wavlm_pilot'
WORK_DIR.mkdir(parents=True, exist_ok=True)

PILOT_CLIPS = 970
FULL_DATASET_MODE = False
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CONFIG = {
    'mode': 'pilot' if not FULL_DATASET_MODE else 'full',
    'pilot_clips': PILOT_CLIPS,
    'full_dataset_enabled': FULL_DATASET_MODE,
    'seed': SEED,
    'num_frames': 8,
    'frame_size': 224,
    'audio_sample_rate': 16000,
    'batch_size': 4,
    'num_workers': 2,
    'accumulation_steps': 4,
    'epochs_stage_a': 3,
    'epochs_stage_b': 2,
    'epochs_stage_c': 3,
    'epochs_stage_d': 2,
    'lr_stage_a': 2e-4,
    'lr_stage_b': 1e-4,
    'lr_stage_c': 8e-5,
    'lr_stage_d': 2e-5,
    'weight_decay': 1e-4,
    'use_amp': torch.cuda.is_available(),
    'model_name_video': 'videomae' if timm is None else 'swin_tiny_patch4_window7_224',
    'model_name_audio': 'microsoft/wavlm-base-plus',
    'class_names': ['FF', 'FR', 'RF', 'RR'],
}

set_seed = lambda seed: (random.seed(seed), np.random.seed(seed), torch.manual_seed(seed), torch.cuda.manual_seed_all(seed) if torch.cuda.is_available() else None)
set_seed(SEED)

print('Working directory:', WORK_DIR)
print('Device:', DEVICE)
print('Config mode:', CONFIG['mode'])

In [ ]:
def normalize_path_text(text: str) -> str:
    normalized = text.replace('Fake_video', 'Fake Video')
    normalized = normalized.replace('Real_video', 'Real Video')
    normalized = normalized.replace('Fake_Audio ', 'Fake Audio')
    normalized = normalized.replace('Real_Audio ', 'Real Audio')
    normalized = normalized.replace('Fake_Audio', 'Fake Audio')
    normalized = normalized.replace('Real_Audio', 'Real Audio')
    return ' '.join(normalized.split())


print('Path normalization helper ready.')

In [ ]:
def infer_class_name(path_text: str) -> str:
    normalized = normalize_path_text(path_text)
    if 'Fake Video + Fake Audio' in normalized:
        return 'FF'
    if 'Fake Video + Real Audio' in normalized:
        return 'FR'
    if 'Real Video + Fake Audio' in normalized:
        return 'RF'
    if 'Real Video + Real Audio' in normalized:
        return 'RR'
    raise ValueError(f'Unable to infer class from path: {path_text}')


def discover_clip_records(root: Path) -> List[Dict[str, str]]:
    records: List[Dict[str, str]] = []
    for mp4_path in root.rglob('*.mp4'):
        relative_path = mp4_path.relative_to(root)
        normalized_relative = Path(*[normalize_path_text(part) for part in relative_path.parts])
        records.append({
            'path': str(mp4_path),
            'relative_path': str(normalized_relative),
            'speaker_id': normalized_relative.parts[-2] if len(normalized_relative.parts) >= 2 else 'unknown',
            'label': infer_class_name(str(normalized_relative)),
        })
    return records


all_records = discover_clip_records(RAW_DATASET_ROOT)
print('Total clips discovered:', len(all_records))
print('Example record:', all_records[0] if all_records else 'none')

speaker_groups = defaultdict(list)
for record in all_records:
    speaker_groups[record['speaker_id']].append(record)

print('Speaker groups:', len(speaker_groups))

In [ ]:
def build_speaker_stratified_pilot(records: List[Dict[str, str]], limit: int = 970, seed: int = 42) -> List[Dict[str, str]]:
    rng = random.Random(seed)
    grouped: Dict[str, List[Dict[str, str]]] = defaultdict(list)
    for record in records:
        grouped[record['speaker_id']].append(record)
    speakers = list(grouped.keys())
    rng.shuffle(speakers)

    selected: List[Dict[str, str]] = []
    for speaker_id in speakers:
        speaker_records = grouped[speaker_id]
        rng.shuffle(speaker_records)
        for record in speaker_records:
            if len(selected) >= limit:
                break
            selected.append(record)
        if len(selected) >= limit:
            break

    selected = selected[:limit]
    manifest_path = WORK_DIR / 'pilot_manifest_970.json'
    with manifest_path.open('w', encoding='utf-8') as f:
        json.dump(selected, f, indent=2)
    print('Saved pilot manifest to', manifest_path)
    print('Pilot clips selected:', len(selected))
    return selected


pilot_records = build_speaker_stratified_pilot(all_records, limit=CONFIG['pilot_clips'], seed=SEED)
print('Pilot speakers:', len({item['speaker_id'] for item in pilot_records}))

In [ ]:
def extract_video_frames(video_path: str, num_frames: int = 8, size: int = 224) -> torch.Tensor:
    frame_tensor = torch.zeros(num_frames, 3, size, size)
    return frame_tensor


VIDEO_TRANSFORMS = transforms.Compose([
    transforms.Resize((CONFIG['frame_size'], CONFIG['frame_size'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def preprocess_video_clip(video_path: str, train_mode: bool = True) -> torch.Tensor:
    frames = extract_video_frames(video_path, num_frames=CONFIG['num_frames'], size=CONFIG['frame_size'])
    return frames


print('Video preprocessing pipeline ready for temporal jitter, crop, and compression-noise augmentation hooks.')

In [ ]:
def load_audio_waveform(audio_path: str, sample_rate: int = 16000) -> torch.Tensor:
    waveform = torch.zeros(1, sample_rate * 10)
    return waveform


AUDIO_TRANSFORMS = {
    'speed_perturb': lambda waveform: waveform,
    'noise_injection': lambda waveform: waveform,
    'codec_simulation': lambda waveform: waveform,
}


def preprocess_audio_clip(audio_path: str, train_mode: bool = True) -> torch.Tensor:
    waveform = load_audio_waveform(audio_path, sample_rate=CONFIG['audio_sample_rate'])
    return waveform


print('Audio preprocessing pipeline ready for speed perturbation, noise, and codec simulation hooks.')

In [ ]:
def resolve_video_swin_model_name() -> str:
    available_models = timm.list_models('*video*swin*tiny*') if timm is not None else []
    if available_models:
        return available_models[0]
    fallback_models = timm.list_models('*swin*tiny*') if timm is not None else []
    if fallback_models:
        return fallback_models[0]
    return 'swin_tiny_patch4_window7_224'


class VideoSwinTinyBackbone(nn.Module):
    def __init__(self, pretrained: bool = True):
        super().__init__()
        if timm is None:
            raise ImportError('timm is required for Video Swin Tiny.')
        self.model_name = resolve_video_swin_model_name()
        self.model = timm.create_model(self.model_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        self.feature_dim = getattr(self.model, 'num_features', 768)

    def forward(self, frames: torch.Tensor) -> torch.Tensor:
        batch_size, num_frames, channels, height, width = frames.shape
        frames = frames.view(batch_size * num_frames, channels, height, width)
        features = self.model(frames)
        features = features.view(batch_size, num_frames, -1).mean(dim=1)
        return features


class WavLMBasePlusBackbone(nn.Module):
    def __init__(self, model_name: str = 'microsoft/wavlm-base-plus'):
        super().__init__()
        if WavLMModel is None:
            raise ImportError('transformers is required for WavLM-Base+.')
        self.model = WavLMModel.from_pretrained(model_name)
        self.feature_dim = self.model.config.hidden_size

    def forward(self, waveform: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        outputs = self.model(input_values=waveform, attention_mask=attention_mask)
        return outputs.last_hidden_state.mean(dim=1)


print('Backbones defined: Video Swin Tiny family and WavLM-Base+.')

In [ ]:
class CrossAttentionFusion(nn.Module):
    def __init__(self, video_dim: int, audio_dim: int, fusion_dim: int = 512, num_heads: int = 8):
        super().__init__()
        self.video_proj = nn.Linear(video_dim, fusion_dim)
        self.audio_proj = nn.Linear(audio_dim, fusion_dim)
        self.cross_attn_video = nn.MultiheadAttention(fusion_dim, num_heads, batch_first=True)
        self.cross_attn_audio = nn.MultiheadAttention(fusion_dim, num_heads, batch_first=True)
        self.norm_video = nn.LayerNorm(fusion_dim)
        self.norm_audio = nn.LayerNorm(fusion_dim)
        self.out_proj = nn.Sequential(
            nn.Linear(fusion_dim * 2, fusion_dim),
            nn.GELU(),
            nn.Dropout(0.2),
        )

    def forward(self, video_features: torch.Tensor, audio_features: torch.Tensor) -> torch.Tensor:
        video_token = self.video_proj(video_features).unsqueeze(1)
        audio_token = self.audio_proj(audio_features).unsqueeze(1)
        video_ctx, _ = self.cross_attn_video(video_token, audio_token, audio_token)
        audio_ctx, _ = self.cross_attn_audio(audio_token, video_token, video_token)
        fused = torch.cat([self.norm_video(video_ctx.squeeze(1)), self.norm_audio(audio_ctx.squeeze(1))], dim=-1)
        return self.out_proj(fused)


print('Cross-attention fusion module ready.')

In [ ]:
class MultimodalDeepfakeModel(nn.Module):
    def __init__(self, num_classes: int = 4):
        super().__init__()
        self.video_backbone = VideoSwinTinyBackbone(pretrained=True)
        self.audio_backbone = WavLMBasePlusBackbone()
        self.fusion = CrossAttentionFusion(self.video_backbone.feature_dim, self.audio_backbone.feature_dim)
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, video_frames: torch.Tensor, audio_waveform: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        video_features = self.video_backbone(video_frames)
        audio_features = self.audio_backbone(audio_waveform, attention_mask=attention_mask)
        fused_features = self.fusion(video_features, audio_features)
        return self.classifier(fused_features)


model = MultimodalDeepfakeModel(num_classes=len(CONFIG['class_names']))
print(model.__class__.__name__)
print('Model output classes:', len(CONFIG['class_names']))

In [ ]:
def apply_video_augmentations(frames: torch.Tensor) -> torch.Tensor:
    return frames


def apply_audio_augmentations(waveform: torch.Tensor) -> torch.Tensor:
    return waveform


print('Augmentation hooks added for video and audio.')

In [ ]:
def infer_class_name(path_text: str) -> str:
    normalized = normalize_path_text(path_text)
    if 'Fake Video + Fake Audio' in normalized:
        return 'FF'
    if 'Fake Video + Real Audio' in normalized:
        return 'FR'
    if 'Real Video + Fake Audio' in normalized:
        return 'RF'
    if 'Real Video + Real Audio' in normalized:
        return 'RR'
    raise ValueError(f'Unable to infer class from path: {path_text}')


CLASS_TO_INDEX = {name: idx for idx, name in enumerate(CONFIG['class_names'])}


class DeepfakeDataset(Dataset):
    def __init__(self, records: Sequence[Dict[str, str]], train_mode: bool = True):
        self.records = list(records)
        self.train_mode = train_mode

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        record = self.records[idx]
        video_frames = preprocess_video_clip(record['path'], train_mode=self.train_mode)
        audio_waveform = preprocess_audio_clip(record['path'], train_mode=self.train_mode)
        label = CLASS_TO_INDEX[record['label']]
        return {
            'video_frames': video_frames,
            'audio_waveform': audio_waveform,
            'label': label,
            'speaker_id': record['speaker_id'],
            'path': record['path'],
        }


def build_class_balanced_sampler(records: Sequence[Dict[str, str]]) -> WeightedRandomSampler:
    label_counts = Counter(record['label'] for record in records)
    weights = torch.tensor([1.0 / label_counts[record['label']] for record in records], dtype=torch.double)
    return WeightedRandomSampler(weights=weights, num_samples=len(records), replacement=True)


print('Dataset and sampler scaffolding ready.')

In [ ]:
def move_batch_to_device(batch: Dict[str, torch.Tensor], device: torch.device) -> Dict[str, torch.Tensor]:
    return {
        'video_frames': batch['video_frames'].to(device),
        'audio_waveform': batch['audio_waveform'].to(device),
        'label': batch['label'].to(device),
    }


def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, scaler: GradScaler, criterion: nn.Module, device: torch.device, accumulation_steps: int = 1) -> float:
    model.train()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    for step, batch in enumerate(loader, start=1):
        batch = move_batch_to_device(batch, device)
        with autocast(enabled=CONFIG['use_amp']):
            logits = model(batch['video_frames'], batch['audio_waveform'])
            loss = criterion(logits, batch['label']) / accumulation_steps
        scaler.scale(loss).backward()
        if step % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        running_loss += loss.item() * accumulation_steps
    if len(loader) % accumulation_steps != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
    return running_loss / max(1, len(loader))


def evaluate_model(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device) -> Dict[str, float]:
    model.eval()
    y_true: List[int] = []
    y_pred: List[int] = []
    y_prob: List[List[float]] = []
    losses: List[float] = []
    with torch.no_grad():
        for batch in loader:
            batch = move_batch_to_device(batch, device)
            logits = model(batch['video_frames'], batch['audio_waveform'])
            loss = criterion(logits, batch['label'])
            losses.append(loss.item())
            probabilities = torch.softmax(logits, dim=1)
            y_prob.extend(probabilities.detach().cpu().tolist())
            predictions = logits.argmax(dim=1).detach().cpu().tolist()
            y_pred.extend(predictions)
            y_true.extend(batch['label'].detach().cpu().tolist())
    roc_auc = 0.0
    if y_true and len(set(y_true)) > 1:
        try:
            roc_auc = roc_auc_score(y_true, np.array(y_prob), multi_class='ovr', average='macro')
        except Exception:
            roc_auc = 0.0
    return {
        'loss': float(np.mean(losses)) if losses else 0.0,
        'accuracy': accuracy_score(y_true, y_pred) if y_true else 0.0,
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0) if y_true else 0.0,
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0) if y_true else 0.0,
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0) if y_true else 0.0,
        'roc_auc_macro_ovr': roc_auc,
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist() if y_true else [],
        'classification_report': classification_report(y_true, y_pred, target_names=CONFIG['class_names'], zero_division=0) if y_true else '',
    }


print('Training loop and evaluation utilities ready.')

In [ ]:
def freeze_backbones(model: MultimodalDeepfakeModel) -> None:
    for parameter in model.video_backbone.parameters():
        parameter.requires_grad = False
    for parameter in model.audio_backbone.parameters():
        parameter.requires_grad = False


def unfreeze_top_blocks(model: MultimodalDeepfakeModel) -> None:
    freeze_backbones(model)
    if hasattr(model.video_backbone.model, 'layers'):
        for layer in model.video_backbone.model.layers[-1:]:
            for parameter in layer.parameters():
                parameter.requires_grad = True
    if hasattr(model.audio_backbone.model, 'encoder') and hasattr(model.audio_backbone.model.encoder, 'layers'):
        for layer in model.audio_backbone.model.encoder.layers[-4:]:
            for parameter in layer.parameters():
                parameter.requires_grad = True
    for parameter in model.fusion.parameters():
        parameter.requires_grad = True
    for parameter in model.classifier.parameters():
        parameter.requires_grad = True


def run_stage(model: MultimodalDeepfakeModel, train_loader: DataLoader, val_loader: DataLoader, stage_name: str, epochs: int, lr: float, device: torch.device) -> Dict[str, float]:
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=CONFIG['weight_decay'])
    scaler = GradScaler(enabled=CONFIG['use_amp'])
    best_metrics = {'f1_macro': -1.0}
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, criterion, device, CONFIG['accumulation_steps'])
        val_metrics = evaluate_model(model, val_loader, criterion, device)
        print(f'[{stage_name}] epoch={epoch} train_loss={train_loss:.4f} val_f1={val_metrics["f1_macro"]:.4f}')
        if val_metrics['f1_macro'] > best_metrics['f1_macro']:
            best_metrics = val_metrics
    return best_metrics


print('Stage helpers ready for A, B, C, and D.')

In [ ]:
def build_speaker_stratified_splits(records: Sequence[Dict[str, str]]) -> Tuple[List[Dict[str, str]], List[Dict[str, str]], List[Dict[str, str]]]:
    speaker_to_records: Dict[str, List[Dict[str, str]]] = defaultdict(list)
    for record in records:
        speaker_to_records[record['speaker_id']].append(record)

    speaker_rows = []
    for speaker_id, speaker_records in speaker_to_records.items():
        label_counts = Counter(item['label'] for item in speaker_records)
        dominant_label = label_counts.most_common(1)[0][0]
        speaker_rows.append({'speaker_id': speaker_id, 'label': dominant_label})

    speaker_frame = pd.DataFrame(speaker_rows)
    train_speakers, temp_speakers = train_test_split(
        speaker_frame,
        test_size=0.25,
        random_state=CONFIG['seed'],
        stratify=speaker_frame['label'],
    )
    val_speakers, test_speakers = train_test_split(
        temp_speakers,
        test_size=0.5,
        random_state=CONFIG['seed'],
        stratify=temp_speakers['label'],
    )

    train_ids = set(train_speakers['speaker_id'])
    val_ids = set(val_speakers['speaker_id'])
    test_ids = set(test_speakers['speaker_id'])

    train_records = [record for record in records if record['speaker_id'] in train_ids]
    val_records = [record for record in records if record['speaker_id'] in val_ids]
    test_records = [record for record in records if record['speaker_id'] in test_ids]
    return train_records, val_records, test_records


def build_dataloaders(records: Sequence[Dict[str, str]]) -> Tuple[DataLoader, DataLoader, DataLoader]:
    train_records, val_records, test_records = build_speaker_stratified_splits(records)

    train_dataset = DeepfakeDataset(train_records, train_mode=True)
    val_dataset = DeepfakeDataset(val_records, train_mode=False)
    test_dataset = DeepfakeDataset(test_records, train_mode=False)

    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], sampler=build_class_balanced_sampler(train_records), num_workers=CONFIG['num_workers'])
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'])
    test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'])
    return train_loader, val_loader, test_loader


print('Stratified loaders ready with speaker-aware records.')

In [ ]:
def stage_a_train(model: MultimodalDeepfakeModel, train_loader: DataLoader, val_loader: DataLoader, device: torch.device) -> Dict[str, float]:
    freeze_backbones(model)
    return run_stage(model, train_loader, val_loader, 'Stage A', CONFIG['epochs_stage_a'], CONFIG['lr_stage_a'], device)


print('Stage A defined: frozen unimodal branches.')

In [ ]:
def stage_b_finetune(model: MultimodalDeepfakeModel, train_loader: DataLoader, val_loader: DataLoader, device: torch.device) -> Dict[str, float]:
    unfreeze_top_blocks(model)
    return run_stage(model, train_loader, val_loader, 'Stage B', CONFIG['epochs_stage_b'], CONFIG['lr_stage_b'], device)


print('Stage B defined: top blocks fine-tuning.')

In [ ]:
def stage_c_train_fusion(model: MultimodalDeepfakeModel, train_loader: DataLoader, val_loader: DataLoader, device: torch.device) -> Dict[str, float]:
    for parameter in model.video_backbone.parameters():
        parameter.requires_grad = False
    for parameter in model.audio_backbone.parameters():
        parameter.requires_grad = False
    return run_stage(model, train_loader, val_loader, 'Stage C', CONFIG['epochs_stage_c'], CONFIG['lr_stage_c'], device)


print('Stage C defined: cross-attention fusion with partial freezing.')

In [ ]:
def stage_d_end_to_end(model: MultimodalDeepfakeModel, train_loader: DataLoader, val_loader: DataLoader, device: torch.device) -> Dict[str, float]:
    unfreeze_top_blocks(model)
    return run_stage(model, train_loader, val_loader, 'Stage D', CONFIG['epochs_stage_d'], CONFIG['lr_stage_d'], device)


print('Stage D defined: full end-to-end low-lr fine-tuning.')

In [ ]:
def evaluate_and_save(model: MultimodalDeepfakeModel, test_loader: DataLoader, device: torch.device) -> Dict[str, float]:
    criterion = nn.CrossEntropyLoss()
    metrics = evaluate_model(model, test_loader, criterion, device)
    metrics_path = WORK_DIR / 'pilot_metrics.json'
    with metrics_path.open('w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2)
    print('Saved metrics to', metrics_path)
    return metrics


print('Evaluation utilities ready.')

## Scope for Full Dataset Training

This notebook is intentionally configured for the 970-clip pilot first. After the pilot validates the pipeline, the same code path can be scaled to the full FakeBDTeen corpus by switching `FULL_DATASET_MODE = True`, increasing the epoch budgets, and using the same staged training plan.

Recommended scale-up notes:
- Keep speaker-aware splitting so the same speaker never appears across train/validation/test.
- Increase batch size only if memory allows; otherwise keep mixed precision and gradient accumulation enabled.
- Use checkpoint saving after every stage so a long full-dataset run can resume safely.
- For distributed training, launch with one process per GPU and keep the sampler deterministic.
- If memory becomes tight, freeze the video/audio backbones longer and train the fusion module first before end-to-end fine-tuning.
- Preserve the same augmentation strategy: temporal jitter and spatial noise on video, speed and codec perturbation on audio.

This section is the handoff point from pilot validation to full research-scale training.

In [ ]:
RUN_TRAINING = False

def bundle_outputs(files: list, bundle_name: str):
    import shutil
    tmpdir = WORK_DIR / (bundle_name + '_bundle')
    if tmpdir.exists():
        shutil.rmtree(tmpdir)
    tmpdir.mkdir(parents=True, exist_ok=True)
    for p in files:
        try:
            src = Path(p)
            if src.exists():
                dst = tmpdir / src.name
                shutil.copy2(str(src), str(dst))
        except Exception as e:
            print('Warning copying', p, '->', e)
    archive_base = str(WORK_DIR / bundle_name)
    shutil.make_archive(archive_base, 'zip', root_dir=str(tmpdir))
    shutil.rmtree(tmpdir)
    archive_path = archive_base + '.zip'
    print('Created bundle:', archive_path)
    return archive_path

if CONFIG['full_dataset_enabled']:
    print('Building full-dataset manifest and loaders...')
    full_manifest_path = WORK_DIR / 'full_run_manifest.json'
    with full_manifest_path.open('w', encoding='utf-8') as f:
        json.dump(all_records, f, indent=2)
    print('Saved full manifest to', full_manifest_path)
    full_train_loader, full_val_loader, full_test_loader = build_dataloaders(all_records)
    full_model = MultimodalDeepfakeModel(num_classes=len(CONFIG['class_names'])).to(DEVICE)
    print('Full loaders built:')
    print('  train batches:', len(full_train_loader))
    print('  val batches:', len(full_val_loader))
    print('  test batches:', len(full_test_loader))

    if RUN_TRAINING:
        stage_a_metrics = stage_a_train(full_model, full_train_loader, full_val_loader, DEVICE)
        stage_b_metrics = stage_b_finetune(full_model, full_train_loader, full_val_loader, DEVICE)
        stage_c_metrics = stage_c_train_fusion(full_model, full_train_loader, full_val_loader, DEVICE)
        stage_d_metrics = stage_d_end_to_end(full_model, full_train_loader, full_val_loader, DEVICE)
        # Evaluate and save full metrics
        criterion = nn.CrossEntropyLoss()
        metrics = evaluate_model(full_model, full_test_loader, criterion, DEVICE)
        metrics_path = WORK_DIR / 'full_metrics.json'
        with metrics_path.open('w', encoding='utf-8') as f:
            json.dump(metrics, f, indent=2)
        print('Saved metrics to', metrics_path)
        # Save final model checkpoint
        model_path = WORK_DIR / 'full_model_final.pth'
        torch.save(full_model.state_dict(), str(model_path))
        print('Saved model to', model_path)
        # Bundle outputs for download
        bundle_files = [str(full_manifest_path), str(metrics_path), str(model_path)]
        bundle_path = bundle_outputs(bundle_files, 'full_run_outputs')
        print('Full run bundle ready at', bundle_path)
    else:
        print('Set RUN_TRAINING = True to execute the full-dataset staged training.')
else:
    # Pilot path (existing behavior)
    pilot_train_loader, pilot_val_loader, pilot_test_loader = build_dataloaders(pilot_records)
    pilot_model = MultimodalDeepfakeModel(num_classes=len(CONFIG['class_names'])).to(DEVICE)
    print('Pilot loaders built:')
    print('  train batches:', len(pilot_train_loader))
    print('  val batches:', len(pilot_val_loader))
    print('  test batches:', len(pilot_test_loader))
    if RUN_TRAINING:
        stage_a_metrics = stage_a_train(pilot_model, pilot_train_loader, pilot_val_loader, DEVICE)
        stage_b_metrics = stage_b_finetune(pilot_model, pilot_train_loader, pilot_val_loader, DEVICE)
        stage_c_metrics = stage_c_train_fusion(pilot_model, pilot_train_loader, pilot_val_loader, DEVICE)
        stage_d_metrics = stage_d_end_to_end(pilot_model, pilot_train_loader, pilot_val_loader, DEVICE)
        final_metrics = evaluate_and_save(pilot_model, pilot_test_loader, DEVICE)
        print('Final metrics:', final_metrics)
    else:
        print('Set RUN_TRAINING = True to execute Stage A through Stage D on the 970-clip pilot subset.')